# BART — Denoising Sequence-to-Sequence Pre-training (2019)

_Papers · Transformers & LLMs_

**Corrupt text with a noising function, then train an encoder-decoder Transformer to reconstruct the original — one model good at both understanding and generation.**

---

This notebook is a practice scaffold for the **BART — Denoising Sequence-to-Sequence Pre-training (2019)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math
torch.manual_seed(0)

# ===== 1) THE ORACLE: one nn.TransformerEncoderLayer step == hand-written attention + FFN =====
torch.manual_seed(7)
D, H = 8, 1                                            # model dim, 1 head -> easy to hand-check
layer = nn.TransformerEncoderLayer(D, H, dim_feedforward=16, dropout=0.0, batch_first=True)
layer.eval()
x = torch.randn(1, 4, D)                               # (batch=1, seq=4, dim=8)
# -- hand-written self-attention (single head): Q,K,V from in_proj, scaled dot product, then out_proj --
W = layer.self_attn.in_proj_weight; b = layer.self_attn.in_proj_bias
q, k, v = (x @ W.t() + b).chunk(3, dim=-1)
att = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(D), dim=-1)
ctx = att @ v
attn_out = ctx @ layer.self_attn.out_proj.weight.t() + layer.self_attn.out_proj.bias
h = layer.norm1(x + attn_out)                          # residual + layernorm (norm_first=False default)
ff = layer.linear2(F.relu(layer.linear1(h)))           # position-wise feed-forward
mine = layer.norm2(h + ff)                             # residual + layernorm
ref = layer(x)                                         # PyTorch's own forward
print("encoder-layer allclose:", torch.allclose(mine, ref, atol=1e-5))   # True

# ===== 2) recompute the WORKED reconstruction-loss example =====
ps = [0.9, 0.5, 0.4]
total = -sum(math.log(p) for p in ps)
print(f"worked recon prob = {ps[0]*ps[1]*ps[2]:.5f}")                     # 0.18000
print(f"worked total loss = {total:.5f}  (= -ln 0.18 = {-math.log(0.18):.5f})")  # 1.71480

# ===== 3) the tiny BART denoiser (Track B: compose imported Transformer blocks) =====
PAD, BOS, MASK = 0, 1, 2                                # special ids
V0, NTOK = 3, 10                                        # 'real' vocab tokens 3..12
V, L, EMB = V0 + NTOK, 8, 32                            # total vocab, clean length, embed dim
def clean_batch(n):
    return torch.randint(V0, V, (n, L))                # random toy 'sentences'

def infill(x, max_span=3):                              # TEXT INFILLING: span -> single [MASK]
    out = []
    for row in x:
        s = torch.randint(1, max_span + 1, (1,)).item()         # span length (toy Poisson-ish)
        st = torch.randint(0, L - s, (1,)).item()               # span start
        keep = torch.cat([row[:st], torch.tensor([MASK]), row[st + s:]])
        keep = F.pad(keep, (0, L - keep.numel()), value=PAD)    # pad back to length L
        out.append(keep)
    return torch.stack(out)

def token_mask(x, k=2):                                 # PLAIN MASKING: each hidden token -> its own [MASK]
    out = x.clone()
    for row in out:
        idx = torch.randperm(L)[:k]
        row[idx] = MASK
    return out

class TinyBART(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, EMB)
        self.pos = nn.Parameter(torch.randn(1, L + 1, EMB) * 0.02)
        enc = nn.TransformerEncoderLayer(EMB, 4, 64, dropout=0.0, batch_first=True)
        dec = nn.TransformerDecoderLayer(EMB, 4, 64, dropout=0.0, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc, 2)
        self.decoder = nn.TransformerDecoder(dec, 2)
        self.out = nn.Linear(EMB, V)
    def forward(self, corrupted, tgt_in):
        mem = self.encoder(self.emb(corrupted) + self.pos[:, :L])     # BIDIRECTIONAL (no mask)
        T = tgt_in.size(1)
        causal = torch.triu(torch.full((T, T), float('-inf')), 1)    # CAUSAL decoder mask
        dec = self.decoder(self.emb(tgt_in) + self.pos[:, :T], mem, tgt_mask=causal)
        return self.out(dec)

def run(corrupt, steps=400, seed=0):
    torch.manual_seed(seed)
    m = TinyBART(); opt = torch.optim.Adam(m.parameters(), 3e-3); lf = nn.CrossEntropyLoss()
    for s in range(steps):
        x = clean_batch(128)                            # CLEAN original = the target
        cx = corrupt(x)                                 # corrupted input -> encoder
        bos = torch.full((128, 1), BOS)
        tgt_in = torch.cat([bos, x[:, :-1]], 1)         # teacher forcing on the CLEAN target
        loss = lf(m(cx, tgt_in).reshape(-1, V), x.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():                               # held-out reconstruction accuracy
        x = clean_batch(400); cx = corrupt(x)
        bos = torch.full((400, 1), BOS); tgt_in = torch.cat([bos, x[:, :-1]], 1)
        acc = (m(cx, tgt_in).argmax(-1) == x).float().mean().item()
    return acc

# ===== 4) ABLATION: text infilling vs plain token masking (paper Section 4) =====
acc_infill = run(lambda x: infill(x),     400, 0)
acc_mask   = run(lambda x: token_mask(x), 400, 0)
print(f"held-out recon acc  text-infilling : {acc_infill:.3f}")
print(f"held-out recon acc  token-masking  : {acc_mask:.3f}")
# -> both well above chance; reconstruction IS learned (OUR toy run, not the paper's ROUGE)
print("chance level:", round(1 / NTOK, 3))

## Visualize the data & results

_As a tiny BART trains to reconstruct corrupted toy text, does held-out reconstruction accuracy improve over training — and how does text infilling (span -> one mask) compare to plain token masking (each token -> its own mask)?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
PAD, BOS, MASK = 0, 1, 2
V0, NTOK = 3, 10
V, L, EMB = V0 + NTOK, 8, 32
def clean_batch(n): return torch.randint(V0, V, (n, L))
def infill(x, ms=3):
    out = []
    for row in x:
        s = torch.randint(1, ms + 1, (1,)).item(); st = torch.randint(0, L - s, (1,)).item()
        keep = torch.cat([row[:st], torch.tensor([MASK]), row[st + s:]])
        out.append(F.pad(keep, (0, L - keep.numel()), value=PAD))
    return torch.stack(out)
def token_mask(x, k=2):
    out = x.clone()
    for row in out: row[torch.randperm(L)[:k]] = MASK
    return out

class TinyBART(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, EMB); self.pos = nn.Parameter(torch.randn(1, L + 1, EMB) * 0.02)
        self.encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(EMB, 4, 64, 0.0, batch_first=True), 2)
        self.decoder = nn.TransformerDecoder(nn.TransformerDecoderLayer(EMB, 4, 64, 0.0, batch_first=True), 2)
        self.out = nn.Linear(EMB, V)
    def forward(self, cx, ti):
        mem = self.encoder(self.emb(cx) + self.pos[:, :L])
        T = ti.size(1); cm = torch.triu(torch.full((T, T), float('-inf')), 1)
        return self.out(self.decoder(self.emb(ti) + self.pos[:, :T], mem, tgt_mask=cm))

def curve(corrupt, steps=400, seed=0):
    torch.manual_seed(seed)
    m = TinyBART(); opt = torch.optim.Adam(m.parameters(), 3e-3); lf = nn.CrossEntropyLoss(); pts = []
    for s in range(steps):
        x = clean_batch(128); cx = corrupt(x)
        ti = torch.cat([torch.full((128, 1), BOS), x[:, :-1]], 1)
        loss = lf(m(cx, ti).reshape(-1, V), x.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        if s % 50 == 0 or s == steps - 1:
            with torch.no_grad():
                xv = clean_batch(400); cv = corrupt(xv)
                tv = torch.cat([torch.full((400, 1), BOS), xv[:, :-1]], 1)
                pts.append((s, round((m(cv, tv).argmax(-1) == xv).float().mean().item(), 3)))
    return pts

print("text infilling :", curve(lambda x: infill(x)))      # green curve above
print("token masking  :", curve(lambda x: token_mask(x)))  # blue curve above

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Recompute the worked reconstruction loss, but suppose better training raised the masked-span probabilities to $p_2=0.8$ ("cat") and $p_3=0.7$ ("sat"), with $p_1=0.9$ ("the") unchanged. What are the new reconstruction probability and total loss, and which positions improved the loss most?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Reconstruction probability $= 0.9\times 0.8\times 0.7 = 0.504$. — _Multiply the per-token probabilities (autoregressive factorization)._
- Per-token losses: $-\ln 0.9 = 0.10536$, $-\ln 0.8 = 0.22314$, $-\ln 0.7 = 0.35667$. — _Cross-entropy is the negative log of each correct-token probability._
- Total $= 0.10536 + 0.22314 + 0.35667 = 0.68517$ (check: $-\ln 0.504 = 0.68517$). — _Sum of logs equals log of the product._

**Answer:** Reconstruction probability $=0.504$; total loss $=0.68517$, down from $1.71480$. The biggest drops are at the masked-span positions 2 and 3 (loss $0.69315\to 0.22314$ and $0.91629\to 0.35667$), because those are the tokens the model actually had to recover — visible token 1 barely changed. This is why the masked positions are where pre-training learning concentrates.

</details>

**Problem 2.** You have the clean sentence "a b c d e" and want to corrupt it two ways: (i) Text Infilling that hides the span "b c d", and (ii) Token Masking that hides the same three tokens. Write each corrupted sequence (using $M$ for $[\text{MASK}]$), and state what extra thing the model must infer under infilling that it does not under masking.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Text Infilling: the whole span "b c d" becomes a SINGLE mask, so $\tilde{x} = [a,\ M,\ e]$ — length 3. — _Section 2.2: each sampled span is replaced with one $[\text{MASK}]$, regardless of span length._
- Token Masking: each hidden token gets its own mask, so $\tilde{x} = [a,\ M,\ M,\ M,\ e]$ — length 5, unchanged. — _BERT-style masking is position-preserving: one mask per token._
- Compare what the decoder must produce. — _Both must recover b, c, d; only one also hides the count._

**Answer:** Infilling gives $[a, M, e]$; masking gives $[a, M, M, M, e]$. Under infilling the model must additionally infer how many tokens the single mask hides (here 3) — the length information is destroyed. Under token masking the length is preserved, so the model only has to fill known slots. This extra "how many" signal is exactly why the paper found text infilling so effective (Section 4.3).

</details>

**Problem 3.** Ablation (from the CODE): on the toy reconstruction task, train the denoiser once with text infilling (span &rarr; one mask) and once with plain token masking (each token &rarr; its own mask). Predict what happens to held-out reconstruction accuracy and explain it in terms of what each corruption forces the model to learn.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Run both corruptions with the same seed, model, and number of steps. — _Isolates the corruption type — the paper's Section 4 ablation, scaled down._
- Compare the held-out per-token reconstruction accuracy curves printed by the CODE. — _The qualitative gap is the reproducible signal, not a single number._
- Reason: infilling destroys length, so the decoder must also model how many tokens are missing; masking preserves length, an easier slot-filling problem. — _This is the mechanism the paper highlights for why infilling is a stronger pre-training signal._

**Answer:** In our toy run, both corruptions improve over training (reconstruction accuracy climbs), and plain token masking reaches higher held-out accuracy on this toy task because it is the easier, length-preserving problem — see the CODEVIZ curves. The point of the demo is the direction the paper emphasizes: text infilling is the harder objective (the model must also infer span length), which on real corpora makes it a stronger pre-training signal (the paper's Section 4.3: "BART models using text-infilling perform well on all tasks"). On a tiny toy vocabulary the easier objective can look better — these are our numbers, not the paper's ROUGE/BLEU.

</details>